In [9]:
import torch
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
import torchaudio
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import kagglehub
from pathlib import Path

cuda


### TO DO 

#### fix scroll ahh

#### Start with a small synthetics dataset speech + noise, then consider
- Metrics
- Types of transforms
- UNet architecture

# Dataset creation

In [ ]:
class WavDataset(Dataset):
    """
    A dataset that loads lazily WAV files .
    
    Args:
        dataset_dir (str or Path): Path to the dataset folder.
        sample_size (int): Number of samples at maximum present in the datset.
    """
    def __init__(self, dataset_dir, sample_size=None):
        dataset_dir = Path(dataset_dir)

        # Filename list
        self.wav_files = sorted(dataset_dir.glob("*/*.wav"))
        if not self.wav_files:
            raise ValueError(f"No .wav files found inside {dataset_dir}")

        # Set sample size
        if sample_size is not None:
            self.wav_files = self.wav_files[:sample_size]

    def __len__(self):
        return len(self.wav_files)
    
    def __getitem__(self, index):
        wav_path = self.wav_files[index]
        waveform, sample_rate = torchaudio.load(wav_path)
        waveform = waveform[:10*22050]
        filename = wav_path.name
        return waveform, sample_rate, filename

def collate_fn(batch):
    waveforms, sample_rates, filenames = zip(*batch)

    waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True)
    
    sample_rates = torch.tensor(sample_rates)
    # impose same length and assert sample rate
    return waveforms, sample_rates, filenames
        
   

In [29]:
dataset_dir = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")+'/Data/genres_original' 

# test
dataset = DataLoader(
    WavDataset(dataset_dir),
    batch_size=32,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn
)

print(next(iter(dataset)))

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/ren/miniconda3/envs/audio-denoising/lib/python3.13/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/ren/miniconda3/envs/audio-denoising/lib/python3.13/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "/tmp/ipykernel_14947/1757319628.py", line 33, in collate_fn
    waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True)
  File "/home/ren/miniconda3/envs/audio-denoising/lib/python3.13/site-packages/torch/nn/utils/rnn.py", line 482, in pad_sequence
    return torch._C._nn.pad_sequence(
           ~~~~~~~~~~~~~~~~~~~~~~~~~^
        sequences,  # type: ignore[arg-type]
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        padding_side,  

# NN